In [1]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin

# Load all states
DISTRICTS = r"C:\CMIP_data\cmip6\Climada\Projects\ccart-india\ccart\data\boundaries\INDIA_DISTRICTS_FIXED.gpkg"
all_districts = gpd.read_file(DISTRICTS)

# States to check
states_to_check = [
    'BIHAR',
    'UTTAR PRADESH', 
    'UTTARAKHAND',
    'WEST BENGAL',
    'ASSAM',
    'JHARKHAND',
    'PUNJAB',
    'HARYANA',
    # Peninsular states for comparison
    'ANDHRA PRADESH',
    'TELANGANA',
    'ODISHA',
    'MAHARASHTRA',
]

# Load FSI
FSI_PATH = r"C:\CMIP_data\cmip6\Climada\Projects\ccart-india\ccart\flood\outputs\fsi\ccart_floods_fsi_static_chirps_rescaled.tif"

chirps_transform = from_origin(68.0, 37.0, 0.05, 0.05)
chirps_shape = (604, 585)

with rasterio.open(FSI_PATH) as src:
    fsi = src.read(1).astype("float32")
    fsi_on_chirps = np.zeros(chirps_shape, dtype="float32")
    reproject(
        source=fsi,
        destination=fsi_on_chirps,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=chirps_transform,
        dst_crs="EPSG:4326",
        resampling=Resampling.bilinear
    )

fsi_on_chirps = np.where(
    (fsi_on_chirps > 0) & (fsi_on_chirps <= 1.0),
    fsi_on_chirps, np.nan
)

print(f"\n{'State':<20} {'Total':>8} {'FSI cells':>10} {'Coverage':>10} {'Mean FSI':>10}")
print("-" * 65)

for state in states_to_check:
    state_gdf = all_districts[
        all_districts['state'].str.upper() == state
    ].dissolve()
    
    if len(state_gdf) == 0:
        print(f"{state:<20} NOT FOUND")
        continue
    
    state_mask = rasterize(
        [(state_gdf.union_all(), 1)],
        out_shape=chirps_shape,
        transform=chirps_transform,
        fill=0, dtype='uint8'
    )
    
    total = state_mask.sum()
    covered = ((state_mask == 1) & np.isfinite(fsi_on_chirps)).sum()
    pct = covered / total * 100 if total > 0 else 0
    mean_fsi = float(np.nanmean(
        fsi_on_chirps[state_mask == 1]
    )) if covered > 0 else 0
    
    print(f"{state:<20} {total:>8,} {covered:>10,} {pct:>9.1f}% {mean_fsi:>10.4f}")


State                   Total  FSI cells   Coverage   Mean FSI
-----------------------------------------------------------------
BIHAR                   3,381          9       0.3%     0.0552
UTTAR PRADESH           8,765        457       5.2%     0.0666
UTTARAKHAND             2,000          0       0.0%     0.0000
WEST BENGAL             3,127      1,228      39.3%     0.2464
ASSAM                   2,834         35       1.2%     0.3358
JHARKHAND               2,816        745      26.5%     0.0539
PUNJAB                  1,900          0       0.0%     0.0000
HARYANA                 1,638          0       0.0%     0.0000
ANDHRA PRADESH          5,493        890      16.2%     0.1890
TELANGANA               3,818      1,908      50.0%     0.2258
ODISHA                  5,394      3,146      58.3%     0.0937
MAHARASHTRA            10,589      4,539      42.9%     0.0849
